In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.stocks.checkpoints

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

micro_batch_bronze_path = "workspace.stocks.micro_batch_bronze"
micro_batch_silver_path = "workspace.stocks.micro_batch_silver"

df = spark.readStream.table(micro_batch_bronze_path)

df = (df.withColumn("event_time", sf.to_timestamp(sf.col("datetime")))
           .withColumn("ingestao_ts", sf.current_timestamp())
           .withColumn("open", sf.col("open").cast("double"))
           .withColumn("high", sf.col("high").cast("double"))
           .withColumn("low", sf.col("low").cast("double"))
           .withColumn("close", sf.col("close").cast("double"))
           .withColumn("volume", sf.col("volume").cast("long"))
           .drop("datetime")
        ).select(
                "ticker", "event_time","open", "high", "low", "close", "volume","fonte", "ingestao_ts"
        )

print("-----Feita Cast de Tipos-----")

df = df.dropna()
df = df.dropDuplicates(["ticker", "event_time"])

print("-----Removidos Duplicatas e Nulos-----")


df = (df.withColumn("week_year", sf.concat(sf.weekofyear("event_time"), sf.lit("-"), sf.year("event_time")))
      .withColumn("variacao_real", (sf.col("close") - sf.col("open")))
      .withColumn("variacao_percent", (sf.col("variacao_real") / sf.col("open")*100))
      .withColumn("flag_valor_invalido",
                   sf.when(
                           (sf.col("open") <= 0) |
                           (sf.col("high") <= 0) |
                           (sf.col("low") <= 0) |
                           (sf.col("close") <= 0) |
                           (sf.col("volume") < 0), 
                           True
                   ).otherwise(False)
                   ))

df = df.filter(sf.col("flag_valor_invalido") == False)
df = df.drop("flag_valor_invalido")

print("-----Tabela Limpa e Enriquecida-----")

df.writeStream \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Volumes/workspace/stocks/checkpoints/micro_batch") \
    .outputMode("append") \
    .table(micro_batch_silver_path)

print("-----Stream Finalizado-----")

In [0]:
silver_df = spark.read.table(micro_batch_silver_path)

print(f"--Total de linhas: {silver_df.count()}")

for c in silver_df.columns:
    null_count = silver_df.filter(sf.col(c).isNull()).count()
    print(f"-Coluna '{c}': {null_count} nulos")

silver_df.display()